# ModelGate quickstart — check a dataset before you train on it

This is the primary use case `modelgate` is built around: **before training a model, run MGS against your dataset in the same notebook**, and only proceed if it passes.

No server, no Docker, no upload step — just a Python import.

In [ ]:
# pip install modelgate-mgs   # once released — see ../../../ROADMAP.md Fase 7
# For now, from this repo: pip install -e ../  (run from packages/modelgate-core/examples/)

from modelgate import audit

## 1. Your dataset

In real use this would just be a path to your own dataset — a ZIP file, or a plain directory with one subfolder per class (`dataset/cats/`, `dataset/dogs/`, ...).

For this notebook to be runnable standalone (no external download needed), the cell below generates a tiny synthetic dataset on the fly. Skip this cell entirely if you already have a real dataset — just point `DATASET_PATH` at it.

In [ ]:
import os
import random
import tempfile

from PIL import Image


def _make_synthetic_image(seed: int, size=(64, 64)) -> Image.Image:
    """Random per-pixel noise, not a flat color — MGS-0003's perceptual
    hash is DCT-based and needs real structure to be meaningful. See
    conformance/fixtures/generate.py in the main repo for the same idea."""
    rng = random.Random(seed)
    img = Image.new("RGB", size)
    px = img.load()
    for y in range(size[1]):
        for x in range(size[0]):
            px[x, y] = (rng.randrange(256), rng.randrange(256), rng.randrange(256))
    return img


DATASET_PATH = tempfile.mkdtemp(prefix="modelgate_quickstart_")

for label, count in {"cats": 8, "dogs": 8}.items():
    class_dir = os.path.join(DATASET_PATH, label)
    os.makedirs(class_dir, exist_ok=True)
    for i in range(count):
        _make_synthetic_image(seed=hash((label, i)) & 0xFFFF).save(
            os.path.join(class_dir, f"{i}.jpg")
        )

print(f"Synthetic dataset written to: {DATASET_PATH}")

## 2. Run MGS against it

One function call. No upload, no waiting on a queue — it runs in-process.

In [ ]:
report = audit(DATASET_PATH)

print(f"MGS {report.spec_version} — overall verdict: {report.overall_verdict}\n")
for r in report.requirements:
    print(f"  {r.id}: {r.verdict}   {r.metrics}")

## 3. Gate training on the verdict

This is the actual pattern: don't train on a dataset that hasn't passed. `report.overall_verdict` is `"PASS"`, `"FAIL"`, or `"NOT_EVALUATED"` — never a silent default (that's MGS-0000, fail-closed: an empty or unreadable dataset reports `NOT_EVALUATED`/`FAIL`, not a fake passing score).

In [ ]:
if report.overall_verdict != "PASS":
    raise RuntimeError(
        f"Dataset did not pass MGS {report.spec_version} "
        f"(overall_verdict={report.overall_verdict}) — fix the dataset before training."
    )

print("Dataset passed — proceeding to training.")

# your_model.fit(DATASET_PATH, ...)

## 4. Keep the report as evidence (optional)

Every `Report` carries `spec_version`, `tool_version`, and `dataset_hash` — enough for someone else (a paper reviewer, a teammate) to know exactly what was checked, with what, against which exact bytes. Worth saving alongside your training run.

In [ ]:
with open("mgs_report.json", "w") as f:
    f.write(report.to_json())

print("Saved to mgs_report.json")
print(f"dataset_hash: {report.dataset_hash}")